In [2]:
import os
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from pathlib import Path
import PIL.Image as Image
import torch 
import torchvision 
from torchvision import transforms
import pydicom


# Train Series Analysis

Analyze Train series to find subtle patterns

In [20]:
from numpy import floor


train_df = pd.read_csv("Datasets/rsna-knee/train.csv")
train_series_df = pd.read_csv("Datasets/rsna-knee/train_series.csv")
LABELS = [c for c in train_df.columns if c not in ["StudyInstanceUID", "Report"]]
print(f"Training Samples: {len(train_df)}")
print(f"Training Series: {len(train_series_df)} across {train_df['StudyInstanceUID'].nunique()} unique studies")
print(f"Each Study has: {len(train_series_df)/train_df['StudyInstanceUID'].nunique()} series per study on average")
print(f"Labels: {len(LABELS)} \n{LABELS}")

Training Samples: 4407
Training Series: 24371 across 4407 unique studies
Each Study has: 5.530065804402088 series per study on average
Labels: 12 
['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


Each Study has `5.53` series per study on average which means some studies dont have the normal 5 study or some studies have more need to look more into it

In [51]:
# Analyze the series data to find out the amount of slices each study has. 
# This is important because we want to make sure that we have enough slices for each study to train our model effectively. 
# We will filter out any studies that have less than 5 slices, as these may not provide enough information for the model 
# to learn from.
series_with_3_slices = train_series_df.groupby("StudyInstanceUID").filter(lambda x: len(x) <= 3)
series_with_3_slices_count = series_with_3_slices['StudyInstanceUID'].nunique()
series_with_4_slices = train_series_df.groupby("StudyInstanceUID").filter(lambda x: len(x) == 4)
series_with_4_slices_count = series_with_4_slices['StudyInstanceUID'].nunique()
series_with_5_slices = train_series_df.groupby("StudyInstanceUID").filter(lambda x: len(x) == 5)
series_with_5_slices_count = series_with_5_slices['StudyInstanceUID'].nunique()
series_with_6_slices = train_series_df.groupby("StudyInstanceUID").filter(lambda x: len(x) == 6)
series_with_6_slices_count = series_with_6_slices['StudyInstanceUID'].nunique()
series_with_more_than_6_slices = train_series_df.groupby("StudyInstanceUID").filter(lambda x: len(x) > 6)
series_with_more_than_6_slices_count = series_with_more_than_6_slices['StudyInstanceUID'].nunique()
print(f"Studies with 3 slices: {series_with_3_slices_count}")
print(f"Studies with 4 slices: {series_with_4_slices_count}")
print(f"Studies with 5 slices: {series_with_5_slices_count}")
print(f"Studies with 6 slices: {series_with_6_slices_count}")
print(f"Studies with more than 6 slices: {series_with_more_than_6_slices_count}")

Studies with 3 slices: 1
Studies with 4 slices: 675
Studies with 5 slices: 2299
Studies with 6 slices: 698
Studies with more than 6 slices: 734


In [52]:
675 + 2299 + 698 + 734 + 1

4407